In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
# from helper import load_env
# load_env()
from pydantic import BaseModel, Field
from typing import List, Dict, Type
from typing import List, Optional
import os
import yaml

In [2]:
import os, json, time, gc
import logging 

from dotenv import load_dotenv
from IPython.display import HTML, Markdown, Image, Video
from tqdm import tqdm
from openai import OpenAI, AsyncOpenAI
from openai.types.chat import (ChatCompletion, 
                               ChatCompletionChunk,
                               ChatCompletionContentPartTextParam, 
                               ChatCompletionContentPartImageParam,
                               ChatCompletionStreamOptionsParam)
import asyncio
import aiohttp
import pandas as pd
import re


import base64
from PIL import Image
import io

#fix bug with aysncio and jupyter
import nest_asyncio # for langchain async 
nest_asyncio.apply()

In [3]:
import litellm
from litellm import acompletion, completion

### Test LM Studio Connection By Openai API

In [16]:
LM_STUDIO_BASE_URL = "http://localhost:1234/v1"
api_key= "lm-studio"

In [17]:
client = OpenAI(base_url=LM_STUDIO_BASE_URL, api_key=api_key)

# Replace with the exact model name running in LM Studio
model_name = "google/gemma-4-12b" 

In [18]:
ret = client.chat.completions.create(
    model=model_name,
    messages=[
        {"role": "system", "content": "You are a helpful AI coding assistant."},
        {"role": "user", "content": "Explain how to check memory usage in a Jupyter notebook."}
    ],
    temperature=0.7,
)

Markdown(ret.choices[0].message.content)

Depending on whether you want to see the total memory used by your **Python kernel**, the size of a **specific variable (like a DataFrame)**, or a **line-by-line profile**, there are different ways to do this.

Here are the most common methods:

---

### 1. Check the Total Memory Used by the Kernel (`psutil`)
If you want to know how much RAM your current Jupyter notebook is consuming overall, the `psutil` library is the standard tool.

**Installation:**
```bash
pip install psutil
```

**Usage in a cell:**
```python
import psutil
import os

# Get the memory usage of the current process
process = psutil.Process(os.getpid())
mem_bytes = process.memory_info().rss
print(f"Memory Usage: {mem_bytes / (1024 ** 3):.2f} GB")
```

---

### 2. Check Memory of Specific DataFrames (`pandas`)
If you are working with data and want to know how much memory a specific DataFrame is occupying, use the built-in pandas methods.

**Option A: The `.info()` method (Quick overview)**
```python
import pandas as pd
df = pd.DataFrame({'A': range(1000000)})

# This shows total memory used by the dataframe
df.info(memory_usage='deep') 
```

**Option B: The `.memory_usage()` method (Detailed)**
This is useful if you want to see which specific column is consuming the most space.
```python
# Returns memory usage for each column in bytes
print(df.memory_usage(deep=True))

# To get the total sum in Megabytes:
total_mb = df.memory_usage(deep=True).sum() / (1024**2)
print(f"Total Memory: {total_mb:.2f} MB")
```
*Note: Always use `deep=True` when working with strings/objects, otherwise pandas only reports the size of the pointer, not the data.*

---

### 3. Line-by-Line Profiling (`memory_profiler`)
If you have a specific function or loop that is making your notebook slow and you want to see exactly which line is "leaking" memory, use `memory_profiler`.

**Installation:**
```bash
pip install memory_profiler
```

**Usage in Notebook:**
You can use the `%mem_report` magic (if available) or wrap a function with the `@profile` decorator.

```python
%load_ext memory_profiler

def my_function():
    a = [1] * 1000000
    b = [2] * 1000000
    return a, b

%mem_report
my_function()
```

---

### 4. Simple "Quick Check" (No extra libraries)
If you don't want to install anything and just want to see the size of a single object (like a list or a basic array), use `sys`.

**Note:** This is less accurate for complex objects like DataFrames because it doesn't always look "inside" the container.

```python
import sys
my_list = [i for i in range(10000)]
print(f"{sys.getsizeof(my_list) / 1024:.2f} KB")
```

---

### Summary Table: Which one should I use?

| Goal | Best Tool | Why? |
| :--- | :--- | :--- |
| **"Is my notebook crashing the server?"** | `psutil` | Shows total RAM used by the current process. |
| **"How big is this DataFrame?"** | `df.info()` | Standard way to check data size in pandas. |
| **"Which line of code is slow/heavy?"** | `memory_profiler` | Provides a line-by-line breakdown. |
| **"Quick check of one variable."** | `sys.getsizeof()` | Built-in, but less accurate for complex objects. |

### 💡 Pro-Tip: Freeing Memory
If you find that your notebook is using too much memory, you can manually trigger Python's garbage collector after deleting a large variable:

```python
import gc
del large_df  # Delete the object
gc.collect()  # Force the garbage collector to run
```

## Test LM studio LLM Connection by liteLLM API

In [12]:
# Markdown(completion.choices[0].message.content)

In [47]:
# 3. Define the async function
async def get_chat_completion(
    api_base,
    api_key, 
    model_name = "openai/local-model",
    system_prompt= "You are a helpful assistant.",
    user_prompt="",
    temperature= 0.7,
    max_tokens=4096):
    
    response = await acompletion(
        model=model_name,  # Prefix with 'openai/' so LiteLLM uses the OpenAI structure
        api_base=api_base,
        api_key=api_key,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response

In [43]:
%%time
# 4. Execute the async function directly in Jupyter
response = asyncio.run(get_chat_completion(api_base=LM_STUDIO_BASE_URL, 
                                           api_key=api_key,
                                           model_name="openai/local-model",
                                            user_prompt="What is LLM?"))



Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/usr/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x794afd35cec0> is already entered


CPU times: user 7.65 ms, sys: 312 μs, total: 7.96 ms
Wall time: 41.9 s


In [44]:
Markdown(response.choices[0].message.content)



An **LLM** stands for **Large Language Model**. It's a type of artificial intelligence system designed to understand, generate, and work with human language at a very high level.

### 🔍 How It Works (Simplified)
- Built on deep learning neural networks, specifically the **Transformer architecture**.
- Trained on massive amounts of text data (books, websites, code, articles, etc.).
- Learns by predicting the next word/token in a sequence, which helps it internalize grammar, facts, reasoning patterns, tone, and structure.

### ✨ Key Capabilities
- Answer questions & hold multi-turn conversations
- Write essays, emails, stories, or creative content
- Translate languages & summarize long documents
- Generate & debug code
- Perform basic logical/mathematical reasoning (to varying degrees)

### 🌍 Common Examples
OpenAI's **GPT** series, Google's **Gemini**, Anthropic's **Claude**, Meta's **Llama** family, Mistral, and many open-source or specialized variants.

### ⚠️ Important Limitations
- **No true understanding**: LLMs are advanced pattern matchers, not conscious or reasoning entities.
- **Hallucinations**: They can confidently generate incorrect or fabricated information.
- **Bias & safety**: Reflect patterns (including biases) present in training data; require careful alignment and oversight.
- **Knowledge cutoff**: Unless connected to live tools/search, they don't know events after their training data ends.

### 💡 Why It Matters
LLMs are powering AI assistants, search enhancements, coding copilots, content creation tools, research accelerators, and more. They're shifting how humans interact with computers from command-based interfaces to natural language dialogue.

Want a deeper dive into how transformers work, the training process, ethical concerns, or how to use them effectively? Just ask!

## Generate COT Data

In [105]:
def generate_cot_data(prompt: str, answer: str) -> str:
    system_prompt = """You are an expert at discovering hidden transformation rules in Alice's Wonderland puzzles.

Think step by step inside <think> </think> tags.
Focus only on explaining how to discover the hidden rule.
Do NOT output the final answer yourself."""

    user_message = f"""Puzzle:
{prompt}

Correct Answer: {answer}

Please think step by step inside <think> tags about how to discover the transformation rule."""

    try:
        response = asyncio.run (acompletion(
            model="openai/local-model",  # Prefix with 'openai/' so LiteLLM uses the OpenAI structure
            api_base=LM_STUDIO_BASE_URL,
            api_key=api_key,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message}
            ],
            temperature=0.3,
            max_tokens=1600,
            timeout=180
            # reasoning_effort="medium"
        ))
        message = response.choices[0].message
        reasoning = getattr(message, "reasoning_content", "") or ""
        content = message.content or ""

        # Combine reasoning
        if reasoning:
            thinking_part = f"<think>\n{reasoning.strip()}\n</think>"
        else:
            thinking_part = f"<think>\n{content.strip()}\n</think>"

        # === Hardcode the final answer (Most Reliable) ===
        final_output = f"{thinking_part}\n\\boxed{{{answer}}}"

        return final_output
        
    except Exception as e:
        print(f"Error generating CoT for prompt: {e}")
        # Fallback: still return something usable
        return f"<think>\nUnable to generate reasoning.\n</think>\n\\boxed{{{answer}}}"
                            

In [106]:
testFile ="../src/Dataset/test.csv"
trainFile = "../src/Dataset/train.csv"


In [107]:
trainDF = pd.read_csv(trainFile)
trainDF

,id,prompt,answer
0,00066667,"In Alice's Wonderland, a secret bit manipulati...",10010111
1,000b53cf,"In Alice's Wonderland, a secret bit manipulati...",01000011
2,00189f6a,"In Alice's Wonderland, secret encryption rules...",cat imagines book
3,001b24c4,"In Alice's Wonderland, numbers are secretly co...",XXXVIII
4,001c63cb,"In Alice's Wonderland, secret encryption rules...",wizard creates secret
...,...,...,...
9495,ffce9e31,"In Alice's Wonderland, a secret bit manipulati...",01100110
9496,ffd5bada,"In Alice's Wonderland, a secret unit conversio...",32.45
9497,ffd89354,"In Alice's Wonderland, secret encryption rules...",student sees the curious mirror
9498,ffdfb678,"In Alice's Wonderland, secret encryption rules...",the curious mouse creates


In [108]:
# Add new column for CoT reasoning
if "cot_reasoning" not in trainDF.columns:
    trainDF["cot_reasoning"] = ""

In [109]:
trainDF

,id,prompt,answer,cot_reasoning
0,00066667,"In Alice's Wonderland, a secret bit manipulati...",10010111,
1,000b53cf,"In Alice's Wonderland, a secret bit manipulati...",01000011,
2,00189f6a,"In Alice's Wonderland, secret encryption rules...",cat imagines book,
3,001b24c4,"In Alice's Wonderland, numbers are secretly co...",XXXVIII,
4,001c63cb,"In Alice's Wonderland, secret encryption rules...",wizard creates secret,
...,...,...,...,...
9495,ffce9e31,"In Alice's Wonderland, a secret bit manipulati...",01100110,
9496,ffd5bada,"In Alice's Wonderland, a secret unit conversio...",32.45,
9497,ffd89354,"In Alice's Wonderland, secret encryption rules...",student sees the curious mirror,
9498,ffdfb678,"In Alice's Wonderland, secret encryption rules...",the curious mouse creates,


In [110]:
trainFile = "train.csv"                    # your original file
outputFile = "train_cot.csv"               # output file with CoT
# MODEL = "gpt-4o"                         # or "claude-3-5-sonnet-20241022"
DELAY = 0.1                                # seconds between API calls (adjust based on rate limit)

In [112]:
%%time
print("Generating Chain-of-Thought data...")

for idx in tqdm(range(len(trainDF))):
    if trainDF.loc[idx, "cot_reasoning"]:   # Skip if already generated
        continue

    prompt = trainDF.loc[idx, "prompt"]
    answer = str(trainDF.loc[idx, "answer"]).strip()

    cot = generate_cot_data(prompt, answer)
    trainDF.loc[idx, "cot_reasoning"] = cot

    # Save progress every 50 rows (in case of crash)
    if (idx + 1) % 50 == 0:
        trainDF.to_csv(outputFile, index=False)
        print(f"Saved progress at row {idx + 1}")

    time.sleep(DELAY)   # Respect API rate limit

Generating Chain-of-Thought data...


  1%|▍                                     | 101/9500 [00:38<1:00:04,  2.61it/s]

CPU times: user 10.2 ms, sys: 7 μs, total: 10.2 ms
Wall time: 38.7 s


KeyboardInterrupt: 